# Baseline TF-IDF + Regressão Logística (Issue 7) — Colab

Mesma pipeline do `scripts/train_baseline.py`: `fit` do TF-IDF só no treino, seed 42, métricas acurácia/F1-macro/MAE/RMSE. Os CSVs **não** são versionados: faça upload em `data/raw/` na célula 2.

In [ ]:
# Célula 1 — repo + dependências (no Colab clona; localmente pula se já existir)
import os, pathlib
if pathlib.Path('/content').exists() and not pathlib.Path('Projeto-IA-Replica-RatingPrediction').exists():
    !git clone https://github.com/Mur1love/Projeto-IA-Replica-RatingPrediction.git
    %cd Projeto-IA-Replica-RatingPrediction
try:
    import google.colab  # estamos no Colab: NÃO forçar pandas/numpy pinados (quebra google-colab/numba)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    !pip install -q scikit-learn matplotlib nltk joblib
    print('Colab: mantido pandas/numpy do sistema; sklearn/matplotlib/nltk/joblib instalados sem pin')
else:
    !pip install -q -r requirements.txt

In [ ]:
# Célula 2 — subir train.csv e test.csv para data/raw/ (obrigatório: .gitignore não versiona CSVs)
import pathlib
pathlib.Path('data/raw').mkdir(parents=True, exist_ok=True)
pathlib.Path('/content/Projeto-IA-Replica-RatingPrediction/data/raw').mkdir(parents=True, exist_ok=True) if pathlib.Path('/content').exists() else None
try:
    from google.colab import files
    print('Faça upload de train.csv e test.csv')
    up = files.upload()
    for name, data in up.items():
        dest = pathlib.Path('data/raw') / name
        dest.write_bytes(data)
        print(f'salvo {dest} ({len(data)/1e6:.1f} MB)')
except ImportError:
    print('fora do Colab — usando data/raw local')
!ls -lh data/raw/

In [ ]:
# Célula 3 — imports e hiperparâmetros (mesmos defaults do CLI)
import os, pathlib, sys
_root = pathlib.Path.cwd()
REPO = None
for cand in [_root, _root.parent, _root / 'Projeto-IA-Replica-RatingPrediction']:
    if (cand / 'src' / 'preprocessing').exists():
        sys.path.insert(0, str(cand))
        REPO = cand
        break
assert REPO is not None, 'raiz do repo não encontrada'
import numpy as np, pandas as pd, sklearn
from sklearn.model_selection import train_test_split
from src.preprocessing.normalization import normalize_dataframe
from src.preprocessing.encoding import encode_rating
from src.preprocessing.embedding import DEFAULT_PARAMS, build_vectorizer, fit_vectorizer, transform_texts
from src.models.baseline import train_logistic_regression
from src.evaluation.metrics import compute_metrics, majority_class_label, predict_majority
from src.evaluation.plots import save_confusion_matrix
SEED = 42
C = 1.0
MAX_ITER = 1000
SAMPLE_N = int(os.environ.get('SAMPLE_N', '2000'))
print('sklearn', sklearn.__version__, '| SEED', SEED, '| SAMPLE_N', SAMPLE_N, '| TF-IDF', DEFAULT_PARAMS)

In [ ]:
# Célula 4 — treino em AMOSTRA estratificada (rápido no Colab free)
np.random.seed(SEED)
raw_train = pd.read_csv(str(REPO / 'data/raw/train.csv'), usecols=['text', 'rating'])
raw_test = pd.read_csv(str(REPO / 'data/raw/test.csv'), usecols=['text', 'rating'])
train_df = normalize_dataframe(raw_train, 'text', empty='drop')
test_df = normalize_dataframe(raw_test, 'text', empty='drop')
_, tr_sample = train_test_split(train_df, train_size=min(SAMPLE_N, len(train_df)), stratify=train_df['rating'], random_state=SEED)
_, te_sample = train_test_split(test_df, train_size=min(max(200, SAMPLE_N//4), len(test_df)), stratify=test_df['rating'], random_state=SEED)
vec, Xtr = fit_vectorizer(tr_sample['text'], build_vectorizer(**DEFAULT_PARAMS))
Xte = transform_texts(vec, te_sample['text'])
ytr = encode_rating(tr_sample['rating']).to_numpy()
yte = encode_rating(te_sample['rating']).to_numpy()
model = train_logistic_regression(Xtr, ytr, C=C, max_iter=MAX_ITER, random_state=SEED)
print('amostra:', compute_metrics(yte, model.predict(Xte)))
print('majoritária:', compute_metrics(yte, predict_majority(ytr, len(yte))))

In [ ]:
# Célula 5 — treino FULL (41k; ~1–5 min). No Colab rode uma vez; no smoke-test pule com RUN_FULL=0.
import os
if os.environ.get('RUN_FULL', '1') == '1':
    vec, Xtr = fit_vectorizer(train_df['text'], build_vectorizer(**DEFAULT_PARAMS))
    Xte = transform_texts(vec, test_df['text'])
    ytr = encode_rating(train_df['rating']).to_numpy()
    yte = encode_rating(test_df['rating']).to_numpy()
    model = train_logistic_regression(Xtr, ytr, C=C, max_iter=MAX_ITER, random_state=SEED)
    m_lr = compute_metrics(yte, model.predict(Xte))
    m_maj = compute_metrics(yte, predict_majority(ytr, len(yte)))
    print('FULL LR:', m_lr)
    print('FULL majoritária:', m_maj)
    save_confusion_matrix(yte, model.predict(Xte), str(REPO / 'results/figures/confusion_matrix_logreg.png'))
    import json, pathlib
    pathlib.Path(REPO / 'results/metrics').mkdir(parents=True, exist_ok=True)
    pathlib.Path(REPO / 'results/metrics/baseline_logreg.json').write_text(json.dumps({'seed': SEED, 'metrics_logreg': m_lr, 'metrics_majority': m_maj}, indent=2))
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
    ConfusionMatrixDisplay(confusion_matrix(yte, model.predict(Xte)), display_labels=[1,2,3,4,5]).plot(cmap='Blues')
    plt.show()
else:
    print('RUN_FULL=0 — full pulado no smoke-test')

In [ ]:
# Célula 6 — baixar artefatos (só no Colab)
try:
    from google.colab import files
    files.download('results/metrics/baseline_logreg.json')
    files.download('results/figures/confusion_matrix_logreg.png')
except ImportError:
    print('fora do Colab — artefatos já em results/')